# NLLB-200 other-languages: combined model, v2 (gentler recipe)

**Combined/multilingual only - independent per-language training is not repeated here.**
`06_nllb_other_languages.ipynb`'s combined model regressed Somali and Dholuo below their
own zero-shot baseline (chrF2++ -2.29 and -4.21) despite Kiswahili improving (+1.34).
Investigation (see the conversation, not repeated here) found two real causes, and this
notebook addresses both:

1. **`train.py`'s internal evaluation never sets `forced_bos_token_id`, beam search, or
   anti-repetition guards** - confirmed by reading the file directly. For a
   three-target-language combined validation set, that means the in-training "Chrf2pp"
   number (38-40 last time) had no reliable way to know which language a row should even
   decode into - almost certainly why it never resembled any real per-language score.
2. **The checkpoint-selection metric was aggregated across all three languages at once**,
   so it could pick an epoch that was still improving Kiswahili while already hurting
   Somali/Dholuo, without that tradeoff ever being visible in the one number it optimized.

**What changed, and what didn't** (a third-party hyperparameter suggestion was reviewed
and partially adopted - see the conversation for the full comparison):

| | v1 (`06_nllb_other_languages.ipynb`) | v2 (this notebook) |
|---|---|---|
| Learning rate | 5e-5 | **1e-5** - v1's value already caused regression; going *up* (a reviewed suggestion proposed 2e-4) would very likely make it worse, not better |
| LR schedule | none (flat) | **linear, 4% warmup** |
| Eval frequency | once/epoch | **every 500 steps** |
| Eval generation | Trainer default (no forced_bos/beams/repetition guards) | **explicit per-language forced_bos_token_id, beam=4, anti-repetition guards** - same settings Section 9 and `inference.py` already use |
| Stopping | one aggregate chrF2++ across all 3 languages, patience=2 | **per-language chrF2++, stops when Somali OR Dholuo (not an average) fails to beat its own best for 3 checks** - protects the two regressing languages specifically, even if Kiswahili is still improving |
| LoRA rank / target modules | r=16, q_proj/v_proj | **unchanged** - a reviewed suggestion proposed r=32 and adding k_proj/out_proj; rejected, since more adapter capacity is the right move for an *underfitting* model (Ekegusii's situation), not for an already-capable model that's regressing on a small, narrow dataset |
| Training design | combined + 3 independent, compared | **combined only** |

**Not attempted here**: mixing general-domain (non-PSA) data into training, the other
lever discussed - this project's own Ekegusii ablation found "mixed" (broad content)
beat narrow domain-only training, and Somali/Dholuo/Kiswahili have never had that tested,
but it needs data this project doesn't currently have staged for these three languages.
Worth a future iteration.

**New files, nothing overwritten**: `train_other_langs_v2.py` (project root, alongside
`train.py` - not a modification of it) does the actual training/eval/stopping;
`06_nllb_other_languages_v2.ipynb` (this file) is new alongside
`06_nllb_other_languages.ipynb`. `arch_config.py`/`metrics.py`/`inference.py`/
`peft_setup.py` are reused as-is via import, unmodified - their behavior didn't need to
change. All checkpoint/log/deployment names use a `_v2` suffix so nothing here can
collide with or overwrite v1's outputs.

**Data**: `05_other_languages_preprocess.ipynb`'s three outputs, unchanged - same shared
985-row test set, same training pool.

## 1. Setup

In [1]:
!pip install -q torch transformers accelerate sentencepiece evaluate sacrebleu mlflow peft matplotlib pandas datasets


### GPU / CUDA check — run this before anything else

In [2]:
import torch
print(f"torch version       : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU visible to torch - fix this before continuing.")


torch version       : 2.11.0+cu128
CUDA available      : True
GPU                 : NVIDIA H100 PCIe


In [3]:
import sys, os
from pathlib import Path

def find_project_root(marker="arch_config.py"):
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} by walking up from {current}")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"project root: {PROJECT_ROOT}")
if "final_training" not in str(PROJECT_ROOT).lower().replace(" ", "_"):
    print("\n  WARNING: 'Final_Training' doesn't appear in PROJECT_ROOT above - move "
          "this notebook under Final_Training/Notebooks/ before continuing.")

# Reused as-is - just path conventions, unrelated to what changed in this retrain.
from arch_config import CHECKPOINTS_DIR, LOGS_DIR, DEPLOYMENT_DIR, DATA_DIR
import metrics, inference
# NOT train.py - see the title cell for why this retrain needs its own training/eval logic.
from train_other_langs_v2 import train_stage_v2
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed
from datasets import Dataset
import pandas as pd

SEED = 42
set_seed(SEED)
MAX_LEN = 128
CHECKPOINT = "facebook/nllb-200-distilled-600M"   # raw base - same as v1, genuinely independent
OTHER_LANGS_DATA_DIR = Path(DATA_DIR) / "other_langs"

LANGUAGE_CODES = {"kiswahili": "swh_Latn", "somali": "som_Latn", "dholuo": "luo_Latn"}
DOMAINS = ["Agriculture", "Health", "Education", "Security", "Governance"]

# Unchanged from v1 - rejecting the reviewed suggestion's rank-32 / q+k+v+out_proj expansion.
LORA_KWARGS = {"r": 16, "lora_alpha": 32, "lora_dropout": 0.05}

OTHER_LANG_CFG_V2 = {
    "lora_target_modules": ["q_proj", "v_proj"],
    "freeze_layers": 6,
    "unfreeze_embeddings": False,
    "tgt_code": None,
    "related_code": None,
    "learning_rate": 1e-5,   # was 5e-5 in v1 - lowered given the demonstrated forgetting risk
    "batch_size": 32,
}

EVAL_STEPS = 500
PATIENCE = 3
MAX_EPOCHS = 5

# v1's real, Section-9-measured numbers (combined approach) - for direct before/after
# comparison in Section 9 below, without needing to reload and re-score the old checkpoint.
V1_COMBINED_RESULTS = {
    "kiswahili": {"zeroshot": 72.96, "v1_combined": 74.30},
    "somali":    {"zeroshot": 73.48, "v1_combined": 71.19},
    "dholuo":    {"zeroshot": 60.20, "v1_combined": 55.99},
}

print("configured language codes (verified against the real tokenizer next):", LANGUAGE_CODES)
print(f"learning_rate={OTHER_LANG_CFG_V2['learning_rate']}  eval_steps={EVAL_STEPS}  "
     f"patience={PATIENCE}  max_epochs={MAX_EPOCHS}")


project root: /home/jovyan/PSA/Final_Training
configured language codes (verified against the real tokenizer next): {'kiswahili': 'swh_Latn', 'somali': 'som_Latn', 'dholuo': 'luo_Latn'}
learning_rate=1e-05  eval_steps=500  patience=3  max_epochs=5


## 2. Verify the language codes against the real NLLB tokenizer

In [4]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
print("Checking each target language code against the real tokenizer:\n")
vocab = tokenizer.get_vocab()
verified = {}
for lang_name, code_guess in LANGUAGE_CODES.items():
    is_present = code_guess in vocab
    status = "CONFIRMED" if is_present else "NOT FOUND - do not trust this code"
    print(f"  {lang_name:12s} {code_guess:12s} {status}")
    verified[lang_name] = is_present


Checking each target language code against the real tokenizer:

  kiswahili    swh_Latn     CONFIRMED
  somali       som_Latn     CONFIRMED
  dholuo       luo_Latn     CONFIRMED


## 3. Load `05`'s three outputs

In [5]:
train_path = OTHER_LANGS_DATA_DIR / "other_langs_train.csv"
test_path = OTHER_LANGS_DATA_DIR / "other_langs_test.csv"
val_path = OTHER_LANGS_DATA_DIR / "other_langs_validation.csv"
for p in [train_path, test_path, val_path]:
    if not p.exists():
        raise FileNotFoundError(f"{p} not found - run 05_other_languages_preprocess.ipynb first.")

train_pool = pd.read_csv(train_path)
shared_test = pd.read_csv(test_path)
shared_val = pd.read_csv(val_path)
print(f"train pool: {len(train_pool):,} rows")
print(f"shared test set: {len(shared_test):,} rows")
print(f"shared validation set: {len(shared_val):,} rows")


train pool: 19,338 rows
shared test set: 985 rows
shared validation set: 1,589 rows


## 4. Build training records - per language (for validation/test), plus a combined
concatenation for training

`records[lang]` (train/val/test) is used by PerLanguageProtectiveEval directly (it needs
each language's own validation rows to evaluate correctly) and as the source material
concatenated into `combined_train`/`combined_val` for actual training.

In [6]:
def build_records(df, src_col, tgt_col, tgt_code, corpus_label, domain_col=None):
    out = []
    for r in df.itertuples(index=False):
        src, tgt = str(getattr(r, src_col)).strip(), str(getattr(r, tgt_col)).strip()
        if src and src.lower() != "nan" and tgt and tgt.lower() != "nan":
            rec = {"src_lang": "eng_Latn", "tgt_lang": tgt_code, "src": src, "tgt": tgt,
                  "corpus": corpus_label}
            if domain_col is not None:
                rec["domain"] = getattr(r, domain_col)
            out.append(rec)
    return out

records = {}
for lang in ["kiswahili", "somali", "dholuo"]:
    if not verified.get(lang, False):
        print(f"skipping {lang} - code not verified (Section 2)")
        continue
    target_col = lang.capitalize()
    train_rows = build_records(train_pool, "English", target_col, LANGUAGE_CODES[lang], f"{lang}_target")
    val_rows = build_records(shared_val, "English", target_col, LANGUAGE_CODES[lang], f"{lang}_target")
    test_rows = build_records(shared_test, "English", target_col, LANGUAGE_CODES[lang], f"{lang}_target",
                              domain_col="psa_domain")
    records[lang] = (train_rows, val_rows, test_rows)
    print(f"{lang}_target: train={len(train_rows):,} | val={len(val_rows):,} | test={len(test_rows):,}")

combined_train = [r for lang in records for r in records[lang][0]]
combined_val = [r for lang in records for r in records[lang][1]]
val_by_lang = {lang: records[lang][1] for lang in records}   # for PerLanguageProtectiveEval

print(f"\ncombined train (all languages): {len(combined_train):,} rows")
print(f"combined validation (all languages): {len(combined_val):,} rows")


kiswahili_target: train=19,338 | val=1,589 | test=985
somali_target: train=19,338 | val=1,589 | test=985
dholuo_target: train=19,338 | val=1,589 | test=985

combined train (all languages): 58,014 rows
combined validation (all languages): 4,767 rows


## 5. Tokenization - combined only

No per-language tokenization this time - independent training isn't run in this
notebook, so only the combined tensors are needed for `Trainer`. `val_by_lang` above
stays as raw text rows; `PerLanguageProtectiveEval` tokenizes on the fly per generation
call, matching Section 9/`inference.py`'s own pattern exactly.

In [7]:
def nllb_preprocess_combined(batch):
    all_ids = []
    for sl, tl, src, tgt in zip(batch["src_lang"], batch["tgt_lang"], batch["src"], batch["tgt"]):
        tokenizer.src_lang = sl
        tokenizer.tgt_lang = tl
        enc = tokenizer(src, text_target=tgt, max_length=MAX_LEN, truncation=True)
        all_ids.append(enc)
    return {"input_ids": [e["input_ids"] for e in all_ids],
            "attention_mask": [e["attention_mask"] for e in all_ids],
            "labels": [e["labels"] for e in all_ids]}

def tokenize_combined(rows):
    ds = Dataset.from_list(rows)
    return ds.map(nllb_preprocess_combined, batched=True, batch_size=16)

tok_combined_train = tokenize_combined(combined_train)
tok_combined_val = tokenize_combined(combined_val)
print(f"combined tokenization complete - {len(tok_combined_train)} train, {len(tok_combined_val)} val")


Map:   0%|          | 0/58014 [00:00<?, ? examples/s]

Map:   0%|          | 0/4767 [00:00<?, ? examples/s]

combined tokenization complete - 58014 train, 4767 val


## 6. Zero-shot baseline

In [8]:
zeroshot_path = f"{CHECKPOINTS_DIR}/nllb_other_langs_zeroshot"
if not Path(zeroshot_path).exists():
    AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT).save_pretrained(zeroshot_path, safe_serialization=True)
    tokenizer.save_pretrained(zeroshot_path)
    print(f"zero-shot baseline saved: {zeroshot_path}")
else:
    print(f"zero-shot baseline already present (from v1's run): {zeroshot_path}")


zero-shot baseline already present (from v1's run): checkpoints/nllb_other_langs_zeroshot


## 7. Train the combined model - v2 recipe

One call to `train_stage_v2()` - the gentler recipe, per-language-correct evaluation
every `EVAL_STEPS` steps, stops when Somali or Dholuo stops improving. See
`train_other_langs_v2.py` for the full implementation.

In [9]:
combined_path_v2, train_seconds, per_lang_cb = train_stage_v2(
    run_id="nllb_combined_other_langs_v2",
    cfg=OTHER_LANG_CFG_V2,
    stage_name="combined_other_langs_v2",
    init_from=CHECKPOINT,
    train_tok=tok_combined_train,
    val_tok=tok_combined_val,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    lora_kwargs=LORA_KWARGS,
    val_by_lang=val_by_lang,
    language_codes=LANGUAGE_CODES,
    eval_steps=EVAL_STEPS,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    checkpoints_dir=CHECKPOINTS_DIR,
    logs_dir=LOGS_DIR,
)
print(f"\ncombined_v2 checkpoint: {combined_path_v2}")
print(f"per-language best scores during training: {per_lang_cb.best}")


  nllb_combined_other_langs_v2   arch=nllb  stage=combined_other_langs_v2  init=facebook/nllb-200-distilled-600M  use_lora=True  [v2: gentler recipe]
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/nllb_combined_other_langs_v2
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db
  lr=1e-05  eval_steps=500  patience=3  max_epochs=5


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821
  (this transformers version has no warmup_ratio - using warmup_steps=362 instead, ~4% of 9060 total steps)
  schedule -> {'lr_scheduler_type': 'linear', 'warmup_steps': 362}


Step,Training Loss,Validation Loss
500,0.880089,0.585379
1000,0.691341,0.452170
1500,0.669417,0.414438
2000,0.659983,0.395316


perlang_eval_kiswahili_step500      chrF2++=80.17  BLEU=66.97
perlang_eval_somali_step500         chrF2++=85.07  BLEU=70.56
perlang_eval_dholuo_step500         chrF2++=77.80  BLEU=63.91
[per-language eval @ step 500] kiswahili=80.17 | somali=85.07 | dholuo=77.80
perlang_eval_kiswahili_step1000     chrF2++=80.26  BLEU=66.67
perlang_eval_somali_step1000        chrF2++=82.75  BLEU=65.94
perlang_eval_dholuo_step1000        chrF2++=73.58  BLEU=56.44
[per-language eval @ step 1000] kiswahili=80.26 | somali=82.75 | dholuo=73.58
perlang_eval_kiswahili_step1500     chrF2++=80.37  BLEU=66.89
perlang_eval_somali_step1500        chrF2++=83.04  BLEU=66.18
perlang_eval_dholuo_step1500        chrF2++=73.53  BLEU=56.05
[per-language eval @ step 1500] kiswahili=80.37 | somali=83.04 | dholuo=73.53
perlang_eval_kiswahili_step2000     chrF2++=80.30  BLEU=66.94
perlang_eval_somali_step2000        chrF2++=83.27  BLEU=65.94
perlang_eval_dholuo_step2000        chrF2++=73.78  BLEU=57.10
[per-language eval @ st

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


combined_v2 checkpoint: checkpoints/nllb_combined_other_langs_v2/merged
per-language best scores during training: {'somali': 85.0714689686844, 'dholuo': 77.79812981931887}


## 8. Evaluate: zero-shot vs. v1 combined vs. v2 combined, per language and by domain

Full Section-9-style generation (beam=4, correct `forced_bos_token_id`, anti-repetition
guards) over the complete 985-row shared test set - this is the real, final evaluation,
not the cheaper per-language monitoring `PerLanguageProtectiveEval` used during training.
v1's numbers are the ones actually measured in `06_nllb_other_languages.ipynb`'s own
Section 9 (`V1_COMBINED_RESULTS`, Section 1 above) - printed directly alongside v2's for
a clear before/after, without needing to reload and re-score the old checkpoint.

In [10]:
def load_and_generate(path, tgt_code):
    model = AutoModelForSeq2SeqLM.from_pretrained(path)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    tok_ = AutoTokenizer.from_pretrained(path)
    def generate(text):
        tok_.src_lang = "eng_Latn"
        enc = tok_(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
        forced_bos = tok_.convert_tokens_to_ids(tgt_code)
        out = model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LEN,
                             no_repeat_ngram_size=3, repetition_penalty=1.2)
        return tok_.decode(out[0], skip_special_tokens=True)
    return model, generate

eval_results = []
for lang, (_, _, test_rows) in records.items():
    tgt_code = LANGUAGE_CODES[lang]

    for run_name, path in [("zeroshot", zeroshot_path), ("v2_combined", combined_path_v2)]:
        model, generate = load_and_generate(path, tgt_code)

        preds = [generate(r["src"]) for r in test_rows]
        score = metrics.score(preds, [r["tgt"] for r in test_rows], f"nllb_{run_name}_{lang}_overall")
        score.update({"language": lang, "run": run_name, "domain": "ALL", "n": len(test_rows)})
        eval_results.append(score)

        for domain in DOMAINS:
            subset = [r for r in test_rows if r.get("domain") == domain]
            if not subset:
                continue
            preds_d = [generate(r["src"]) for r in subset]
            score_d = metrics.score(preds_d, [r["tgt"] for r in subset], f"nllb_{run_name}_{lang}_{domain}")
            score_d.update({"language": lang, "run": run_name, "domain": domain, "n": len(subset)})
            eval_results.append(score_d)

        del model
        torch.cuda.empty_cache()

results_df = pd.DataFrame(eval_results)[["language", "run", "domain", "n", "chrf2pp", "bleu"]]
results_df.to_csv(f"{LOGS_DIR}/nllb_other_languages_v2_results.csv", index=False)

print("\n=== Before / after, overall (ALL domains) chrF2++ ===")
overall_v2 = results_df[results_df["domain"] == "ALL"].pivot_table(index="language", columns="run", values="chrf2pp")
comparison = overall_v2.copy()
comparison["v1_combined"] = [V1_COMBINED_RESULTS[lang]["v1_combined"] for lang in comparison.index]
comparison = comparison[["zeroshot", "v1_combined", "v2_combined"]]
comparison["v2_vs_zeroshot"] = comparison["v2_combined"] - comparison["zeroshot"]
comparison["v2_vs_v1"] = comparison["v2_combined"] - comparison["v1_combined"]
print(comparison.round(2).to_string())

print("\n=== v2 combined, chrF2++ by domain ===")
by_domain = results_df[(results_df["domain"] != "ALL") & (results_df["run"] == "v2_combined")]
print(by_domain.pivot_table(index="language", columns="domain", values="chrf2pp").round(1).to_string())


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_zeroshot_kiswahili_overall     chrF2++=72.96  BLEU=55.14
nllb_zeroshot_kiswahili_Agriculture chrF2++=73.98  BLEU=56.25
nllb_zeroshot_kiswahili_Health      chrF2++=71.03  BLEU=52.39
nllb_zeroshot_kiswahili_Education   chrF2++=77.85  BLEU=63.23
nllb_zeroshot_kiswahili_Security    chrF2++=75.09  BLEU=57.16
nllb_zeroshot_kiswahili_Governance  chrF2++=71.70  BLEU=53.31


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_v2_combined_kiswahili_overall  chrF2++=74.44  BLEU=56.50
nllb_v2_combined_kiswahili_Agriculture chrF2++=74.22  BLEU=55.47
nllb_v2_combined_kiswahili_Health   chrF2++=72.93  BLEU=54.53
nllb_v2_combined_kiswahili_Education chrF2++=79.35  BLEU=64.96
nllb_v2_combined_kiswahili_Security chrF2++=77.61  BLEU=61.22
nllb_v2_combined_kiswahili_Governance chrF2++=74.14  BLEU=54.92


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

...

nllb_v2_combined_somali_overall     chrF2++=73.07  BLEU=50.32
nllb_v2_combined_somali_Agriculture chrF2++=69.25  BLEU=44.86
nllb_v2_combined_somali_Health      chrF2++=72.40  BLEU=49.30
nllb_v2_combined_somali_Education   chrF2++=77.61  BLEU=57.11
nllb_v2_combined_somali_Security    chrF2++=79.43  BLEU=59.78
nllb_v2_combined_somali_Governance  chrF2++=76.45  BLEU=54.77


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_zeroshot_dholuo_overall        chrF2++=60.20  BLEU=42.10
nllb_zeroshot_dholuo_Agriculture    chrF2++=64.93  BLEU=50.70
nllb_zeroshot_dholuo_Health         chrF2++=56.19  BLEU=35.71
nllb_zeroshot_dholuo_Education      chrF2++=61.19  BLEU=42.23
nllb_zeroshot_dholuo_Security       chrF2++=59.95  BLEU=39.24
nllb_zeroshot_dholuo_Governance     chrF2++=62.08  BLEU=39.20


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

nllb_v2_combined_dholuo_overall     chrF2++=59.56  BLEU=39.51
nllb_v2_combined_dholuo_Agriculture chrF2++=58.98  BLEU=40.47
nllb_v2_combined_dholuo_Health      chrF2++=57.24  BLEU=35.82
nllb_v2_combined_dholuo_Education   chrF2++=62.40  BLEU=43.58
nllb_v2_combined_dholuo_Security    chrF2++=61.77  BLEU=38.88
nllb_v2_combined_dholuo_Governance  chrF2++=65.28  BLEU=43.90

=== Before / after, overall (ALL domains) chrF2++ ===
run        zeroshot  v1_combined  v2_combined  v2_vs_zeroshot  v2_vs_v1
language                                                               
dholuo        60.20        55.99        59.56           -0.64      3.57
kiswahili     72.96        74.30        74.44            1.48      0.14
somali        73.48        71.19        73.07           -0.42      1.88

=== v2 combined, chrF2++ by domain ===
domain     Agriculture  Education  Governance  Health  Security
language                                                       
dholuo            59.0       62.4        65.3

## 9. Save deployment-ready checkpoint

In [11]:
inference.save_for_deployment(combined_path_v2, "nllb", "nllb_combined_other_langs_v2", deployment_dir=DEPLOYMENT_DIR)


deployment-ready: /home/jovyan/PSA/Final_Training/deployment/nllb/nllb_combined_other_langs_v2  (1 safetensors file(s))


PosixPath('deployment/nllb/nllb_combined_other_langs_v2')

## 10. Demonstration

In [12]:
samples = [
    "Farmers are urged to prioritize safe agrochemical usage this season.",
    "Ministry of Health: report suspected cholera cases immediately.",
]

demo_model = AutoModelForSeq2SeqLM.from_pretrained(combined_path_v2)
demo_model.to("cuda" if torch.cuda.is_available() else "cpu")
demo_tok = AutoTokenizer.from_pretrained(combined_path_v2)

for lang in records:
    demo_tok.src_lang = "eng_Latn"
    forced_bos = demo_tok.convert_tokens_to_ids(LANGUAGE_CODES[lang])

    print(f"NLLB (combined v2) - English to {lang.capitalize()} (PSA-adapted)\n")
    for text in samples:
        enc = demo_tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(demo_model.device)
        out = demo_model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LEN,
                                  no_repeat_ngram_size=3, repetition_penalty=1.2)
        print(f"[en] {text}")
        print("  ->", demo_tok.decode(out[0], skip_special_tokens=True))
        print()
del demo_model
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

NLLB (combined v2) - English to Kiswahili (PSA-adapted)

[en] Farmers are urged to prioritize safe agrochemical usage this season.
  -> Wakulima wanahimizwa waweke kipaumbele matumizi salama ya kemikali za kilimo msimu huu.

[en] Ministry of Health: report suspected cholera cases immediately.
  -> Wizara ya Afya: kuripoti visa vya kuaminika kwa kipindupindu mara moja.

NLLB (combined v2) - English to Somali (PSA-adapted)

[en] Farmers are urged to prioritize safe agrochemical usage this season.
  -> Beeraleyda waxaa lagu boorinayaa inay mudnaanta siiyaan isticmaalka kiimikada beeraha ee ammaanka ah xilli ciyaareedkan.

[en] Ministry of Health: report suspected cholera cases immediately.
  -> Wasaaradda Caafimaadka: si degdeg ah u soo sheeg kiisaska laga shakiyo ee cudurka kolera.

NLLB (combined v2) - English to Dholuo (PSA-adapted)

[en] Farmers are urged to prioritize safe agrochemical usage this season.
  -> Ijiwo jopur mondo oket mokwongo tiyo gi gik ma nigi tuoche mag pur e kindew